# Data order tracking

Included currently:

-   AGFD
-   UNSW
-   MCC5


In [ ]:
from pathlib import Path

import numpy as np
import plotly.express as px
import polars as pl
from plotly import graph_objects as go
from scipy.interpolate import CubicSpline, interp1d, InterpolatedUnivariateSpline
from scipy.signal import find_peaks
import matplotlib.pyplot as plt

## Helpers


In [ ]:
# UNSW
def read_UNSW_hive(
    data_location,
    severity=None,
    speed_nominal=None,
    load_nominal=None,
    drop_nominal=True,
):
    """
    data_location: Path object pointing to the hive data directory
    severity options: "H", "S", "M", "L"
    speed_nominal_options: 2, 5, 10, 15, 20
    load_nominal_options: 0, 5, 10, 15, 20
    """

    collect = True
    if isinstance(data_location, pl.DataFrame):
        df_hive = data_location
        collect = False
    elif isinstance(data_location, Path):
        hive_files = list(data_location.glob("**/*.parquet"))
        df_hive = pl.scan_parquet(hive_files, hive_partitioning=True)
    else:
        raise ValueError("data_location must be a polars DataFrame or a Path object")

    df_hive = pl.scan_parquet(
        hive_files,
        hive_partitioning=True,
    )

    # Get only required partitions
    if severity is not None:
        if isinstance(severity, list):
            df_hive = df_hive.filter(
                pl.col("severity").is_in(severity),
            )
        else:
            df_hive = df_hive.filter(
                pl.col("severity") == severity,
            )

    if speed_nominal is not None:
        df_hive = df_hive.filter(
            pl.col("speed_nominal") == speed_nominal,
        )

    if load_nominal is not None:
        df_hive = df_hive.filter(
            pl.col("load_nominal") == load_nominal,
        )

    # Drop nominal values
    if drop_nominal:
        df_hive = df_hive.select(pl.exclude(["load_nominal", "speed_nominal"]))

    if collect:
        df_hive = df_hive.collect()

    return df_hive


# MCC5
def read_MCC5_hive(
    data_location,
    fault=None,
    severity=None,
    circulation=None,
    section=None,
    speed_nominal=None,
    load_nominal=None,
    drop_nominal=True,
):
    """
    data_location: Path object pointing to the hive data directory
    severity options: "H", "S", "M", "L"
    speed_nominal_options: 2, 5, 10, 15, 20
    load_nominal_options: 0, 5, 10, 15, 20
    """

    collect = True
    if isinstance(data_location, pl.DataFrame):
        df_hive = data_location
        collect = False
    elif isinstance(data_location, Path):
        hive_files = list(data_location.glob("**/*.parquet"))
        df_hive = pl.scan_parquet(hive_files, hive_partitioning=True)
    else:
        raise ValueError("data_location must be a polars DataFrame or a Path object")

    # Get only required partitions
    if fault is not None:
        df_hive = df_hive.filter(
            pl.col("fault") == fault,
        )

    if severity is not None:
        df_hive = df_hive.filter(
            pl.col("severity") == severity,
        )

    if circulation is not None:
        df_hive = df_hive.filter(
            pl.col("circulation") == circulation,
        )

    if section is not None:
        df_hive = df_hive.filter(
            pl.col("section") == section,
        )

    if speed_nominal is not None:
        df_hive = df_hive.filter(
            pl.col("speed_nominal") == speed_nominal,
        )

    if load_nominal is not None:
        df_hive = df_hive.filter(
            pl.col("load_nominal") == load_nominal,
        )

    # Drop nominal values
    if drop_nominal:
        df_hive = df_hive.select(pl.exclude(["load_nominal", "speed_nominal"]))

    if collect:
        df_hive = df_hive.collect()

    return df_hive


# AGFD
def read_AGFD_hive(
    data_location,
    fault=None,
    severity=None,
    installation=None,
    GP=None,
    speed_nominal=None,
    load_nominal=None,
    drop_nominal=True,
):
    """
    data_location: Path object pointing to the hive data directory
    severity options: "-", "S", "L"
    speed_nominal_options: 250, 500, 750, 1000, 1500
    load_nominal_options: 1, 6, 11 (%)
    """

    collect = True
    if isinstance(data_location, pl.DataFrame):
        df_hive = data_location
        collect = False
    elif isinstance(data_location, Path):
        hive_files = list(data_location.glob("**/*.parquet"))
        df_hive = pl.scan_parquet(hive_files, hive_partitioning=True)
    else:
        raise ValueError("data_location must be a polars DataFrame or a Path object")

    # Get only required partitions
    if fault is not None:
        df_hive = df_hive.filter(
            pl.col("fault") == fault,
        )

    if severity is not None:
        df_hive = df_hive.filter(
            pl.col("severity") == severity,
        )

    if GP is not None:
        df_hive = df_hive.filter(
            pl.col("GP") == GP,
        )

    if installation is not None:
        df_hive = df_hive.filter(
            pl.col("installation") == installation,
        )

    if speed_nominal is not None:
        df_hive = df_hive.filter(
            pl.col("speed_nominal") == speed_nominal,
        )

    if load_nominal is not None:
        df_hive = df_hive.filter(
            pl.col("load_nominal") == load_nominal,
        )

    # Drop nominal values
    if drop_nominal:
        df_hive = df_hive.select(pl.exclude(["load_nominal", "speed_nominal"]))

    if collect:
        df_hive = df_hive.collect()

    return df_hive

In [ ]:
# # Naive
# def order_track_signal(
#     signal,
#     time,
#     tacho,
#     tacho_threshold,
#     num_samples_per_revolution,
#     tacho_component_ratio=1,
# ):
#     # Get times of crossings
#     # crossings = find_peaks(tacho, prominence=0.5, distance=100)[0]

#     crossings = np.where(
#         np.logical_and(tacho[:-1] <= tacho_threshold, tacho[1:] > tacho_threshold)
#     )[0]

#     # Get time at crossings
#     t_crossings = time[crossings] / tacho_component_ratio

#     # Create angles for each crossing (different anchor to original encoder angles)
#     angle_crossings = np.arange(len(t_crossings)) * 360

#     # Add intermediate sampling point angles
#     interpolation_angles = (
#         np.arange(num_samples_per_revolution * (len(t_crossings) - 1))
#         / num_samples_per_revolution
#         * 360
#     )
#     # Get corresponding times for each new sample point
#     interpolation_times = np.interp(interpolation_angles, angle_crossings, t_crossings)

#     cs = CubicSpline(time, signal)

#     return cs(interpolation_times), interpolation_times

In [ ]:
# MCC5
def order_track_signal(
    signal,
    time,
    tacho,
    tacho_threshold,
    num_samples_per_revolution,
    tacho_component_ratio=1,
):
    # Get times of crossings
    # crossings = find_peaks(tacho, prominence=0.5, distance=100)[0]

    crossings = np.where(
        np.logical_and(tacho[:-1] <= tacho_threshold, tacho[1:] > tacho_threshold)
    )[0]

    # Get time at crossings
    t_crossings = time[crossings] / tacho_component_ratio

    # Create angles for each crossing (different anchor to original encoder angles)
    angle_crossings = np.arange(len(t_crossings)) * 360

    if tacho_component_ratio != 1:
        real_angle_crossings = angle_crossings / tacho_component_ratio
        real_t_crossings = np.interp(real_angle_crossings, angle_crossings, t_crossings)

        t_crossings = real_t_crossings
        angle_crossings = real_angle_crossings

    # Add intermediate sampling point angles
    interpolation_angles = (
        np.arange(num_samples_per_revolution * (len(t_crossings) - 1))
        / num_samples_per_revolution
        * 360
    )
    # Get corresponding times for each new sample point
    interpolation_times = np.interp(interpolation_angles, angle_crossings, t_crossings)

    cs = CubicSpline(time, signal)

    return cs(interpolation_times), interpolation_times, len(angle_crossings) - 1


# https://github.com/andrek10/bearing-vibration-diagnostics-toolbox/blob/master/pyvib/signal.py
def ordertrack_pyvib(t, y, t_s, s, ds, cubic=True):
    """
        Order track the vibration signal to match a desired position vector

        Parameters
        ----------
    t : float 1D array
        Time signal
    y : float 1D array
        Signal to order track
    t_s : float 1D array
        Time of position signal
    s : float 1D array
        Position signal in number of rounds
    ds : float
        Desired delta position
    cubic : boolean, optional
        Whether cubic interpolation is used

        Returns
        -------
    s_ot : float 1D array
        Shaft position of returned signal
    y_ot : float 1D array
        Order tracked signal
    """

    # Copy data
    t = np.array(t)
    y = np.array(y)
    t_s = np.array(t_s)
    s = np.array(s)

    # Parameters
    dt = t[1] - t[0]

    # Get the overlapping parts between y and s
    temp = t[0]
    t = t - temp
    t_s = t_s - temp
    i1 = 0
    i2 = t.size
    if t_s[0] > 0.0:
        i1 = int(np.ceil(t_s[0] / dt))
    if t_s[-1] < t[-1]:
        i2 = int(np.floor(t_s[-1] / dt))
    t = t[i1:i2]
    y = y[i1:i2]

    # Interpolate the measured position to match vibration time
    f = interp1d(t_s, s)
    s = f(t)

    # Make desired position signal
    s_ot = np.arange(s[0], s[-1], ds)

    # print('s.size=%i, y.size=%i' % (s.size, y.size))

    # Make order tracked signal
    if cubic is True:
        f = InterpolatedUnivariateSpline(s, y, k=3)
    else:
        f = interp1d(s, y)
    y_ot = f(s_ot)

    # Return data
    return s_ot, y_ot

In [ ]:
# AGFD
def order_track_signal_angle(
    signal,
    time,
    angle,
    num_samples_per_revolution,
):
    # Constants
    encoder_target = 5  # * Arbitrarily chosen key revolution starting point

    # Wrap the encoder angle (degrees)
    encoder_wrapped = angle % 360

    # Get times of crossings
    crossings = np.where(
        np.logical_and(
            encoder_wrapped[:-1] <= encoder_target, encoder_wrapped[1:] > encoder_target
        )
    )[0]

    # Get accurate times for crossings
    t_crossings = time[crossings] + (encoder_target - encoder_wrapped[crossings]) / (
        encoder_wrapped[crossings + 1] - encoder_wrapped[crossings]
    ) * (time[crossings + 1] - time[crossings])

    # Create angles for each crossing (different anchor to original encoder angles)
    angle_crossings = np.arange(len(t_crossings)) * 360

    # Add intermediate sampling point angles
    interpolation_angles = (
        np.arange(num_samples_per_revolution * (len(t_crossings) - 1))
        / num_samples_per_revolution
        * 360
    )
    # Get corresponding times for each new sample point
    interpolation_times = np.interp(interpolation_angles, angle_crossings, t_crossings)

    cs = CubicSpline(time, signal)

    return cs(interpolation_times)

## UNSW

-   Mendeley Data: https://data.mendeley.com/datasets/p72x3m92cv/1

**Notes**

-   Sampling frequency: 200 kHz
-   "S" size crack doesn't really show any fault signs


In [ ]:
sampling_freq_UNSW = 200000
num_teeth_UNSW = 27

speed_nominal_UNSW = 10

In [ ]:
UNSW_processed_data_dir = Path.cwd().parent.parent.parent / "data" / "ROTOMATE" / "UNSW"

# # Read everything

# read_UNSW_hive(UNSW_processed_data_dir / "UNSW_gear_crack_hive")
# # Get one thing

# read_UNSW_hive(
#     UNSW_processed_data_dir / "UNSW_gear_crack_hive",
#     severity="H",
#     speed_nominal=10,
#     load_nominal=10,
# ).head(3)

# # Loop through all
# for severity in ["H", "S", "M", "L"]:
#     for speed_nominal in [2, 5, 10, 15, 20]:
#         for load_nominal in [0, 5, 10, 15, 20]:
#             measurement = read_UNSW_hive(
#                 UNSW_processed_data_dir / "UNSW_gear_crack_hive",
#                 severity=severity,
#                 speed_nominal=speed_nominal,
#                 load_nominal=load_nominal,
#             )

In [ ]:
def plot_UNSW_FFT(df):
    speed_real_UNSW = df["speed"][0] / 60  # Hz
    GMF_UNSW = speed_real_UNSW * num_teeth_UNSW  # Hz

    sig = df["signal"]
    y = np.abs(np.fft.rfft(sig))
    x = np.fft.rfftfreq(len(sig), d=1 / sampling_freq_UNSW)

    x = x[: len(x) // 2**6]
    y = y[: len(y) // 2**6]

    fig = px.line(x=x, y=y, labels={"x": "Frequency (Hz)", "y": "Amplitude"})

    # GMF
    for i in range(1, 6):
        fig.add_vline(x=GMF_UNSW * i, line_dash="dash", line_color="green")

        for j in range(1, 3):
            fig.add_vline(
                x=GMF_UNSW * i - speed_real_UNSW * j, line_dash="dot", line_color="blue"
            )
            fig.add_vline(
                x=GMF_UNSW * i + speed_real_UNSW * j, line_dash="dot", line_color="blue"
            )

    return fig

In [ ]:
mes_M_UNSW = read_UNSW_hive(
    UNSW_processed_data_dir / "UNSW_gear_crack_hive",
    severity="M",
    speed_nominal=speed_nominal_UNSW,
    load_nominal=10,
)[: sampling_freq_UNSW * 2]

mes_L_UNSW = read_UNSW_hive(
    UNSW_processed_data_dir / "UNSW_gear_crack_hive",
    severity="L",
    speed_nominal=speed_nominal_UNSW,
    load_nominal=10,
)[: sampling_freq_UNSW * 5]

fig_M = plot_UNSW_FFT(mes_M_UNSW)
fig_L = plot_UNSW_FFT(mes_L_UNSW).update_traces(line_color="red")

go.Figure(data=fig_M.data + fig_L.data).show()

In [ ]:
asd = mes_L_UNSW["tacho_in"][: len(mes_L_UNSW["tacho_in"]) // 16]
# peaks = find_peaks(asd, distance=1000)
peaks = find_peaks(asd, prominence=0.5, distance=100)
fig = go.Figure(
    px.line(asd).data
    + px.scatter(x=peaks[0], y=asd[peaks[0]], color_discrete_sequence=["red"]).data
)
fig.show()

In [ ]:
np.diff(peaks[0])

In [ ]:
OT_L_UNSW, OT_L_UNSW_times = order_track_signal(
    mes_L_UNSW["signal"].to_numpy(),
    np.arange(len(mes_L_UNSW["signal"])) / sampling_freq_UNSW,
    mes_L_UNSW["tacho_in"].to_numpy(),
    tacho_threshold=1,
    num_samples_per_revolution=2700,
    tacho_component_ratio=1,
)

OT_L_UNSW = OT_L_UNSW[: len(OT_L_UNSW) // 16]
OT_L_UNSW_times = OT_L_UNSW_times[: len(OT_L_UNSW_times) // 16]

a = mes_L_UNSW["signal"][: len(mes_L_UNSW) // 16]
a_time = np.arange(len(a)) / sampling_freq_UNSW

fig1 = px.line(x=OT_L_UNSW_times, y=OT_L_UNSW)
fig2 = px.line(x=a_time, y=a).update_traces(line_color="red")

go.Figure(fig1.data + fig2.data).show()

## MCC5

-   Journal paper: https://www.sciencedirect.com/science/article/pii/S2352340924004220
-   Data in Brief: https://arxiv.org/pdf/2403.12521
-   Mendeley Data: https://data.mendeley.com/datasets/p92gj2732w/2
-   GitHub\_ https://github.com/liuzy0708/MCC5-THU-Gearbox-Benchmark-Datasets

**Known problems**

_Suffix `ERROR_` added to these files to make skipping easier\_

-   gear_pitting_M_speed_circulation_20Nm-1000rpm.csv - Distortion, gain error, or something
-   gear_pitting_M_torque_circulation_2000rpm_10Nm.csv - Distortion, gain error, or something

**Notes**

-   Sampling frequency: 12.8 kHz
-   `gearbox_vibration_x` seems to work the best
-   Speeds seem to be +/- 50 RPM from the reported ones, but highest discrepancy is almost 150 RPM (at motor).


In [ ]:
MCC5_processed_data_dir = Path.cwd().parent.parent.parent / "data" / "ROTOMATE" / "MCC5"

sampling_freq_MCC5 = 12800
num_teeth_MCC5 = 36
first_gear_ratio_MCC5 = 29 / 95  # Ratio of first gear in the drivetrain
samples_per_revolution_MCC5 = num_teeth_MCC5 * 10
# samples_per_revolution_MCC5 = num_teeth_MCC5 * 20

sensor_MCC5 = "gearbox_vibration_x"
speed_nominal_MCC5 = 2000
load_nominal_MCC5 = 20

In [ ]:
def plot_MCC5_FFT(
    df, sensor="gearbox_vibration_x", log=False, num_harmonics=8, sideband_block_size=5
):
    speed_real = df["speed"][0] / 60  # Hz
    GMF = speed_real * num_teeth_MCC5  # Hz

    sig = df[sensor]
    y = np.abs(np.fft.rfft(sig))
    x = np.fft.rfftfreq(len(sig), d=1 / sampling_freq_MCC5)

    if log:
        y = np.log(y)

    # x = x[: len(x) // 2**2]
    # y = y[: len(y) // 2**2]

    fig = px.line(x=x, y=y, labels={"x": "Frequency (Hz)", "y": "Amplitude"})

    # GMF
    for i in range(1, num_harmonics + 1):
        fig.add_vline(x=GMF * i, line_dash="dash", line_color="red")

        for j in range(1, sideband_block_size + 1):
            # Inner sidebands
            fig.add_vline(
                x=GMF * i - speed_real * j,
                line_dash="dot",
                line_color="orange",
                line_width=1.5,
            )
            fig.add_vline(
                x=GMF * i + speed_real * j,
                line_dash="dot",
                line_color="firebrick",
                line_width=1.5,
            )

            # Outer sidebands
            fig.add_vline(
                x=GMF * i - speed_real * (18 - j),
                line_dash="dot",
                line_color="orange",
                line_width=1.5,
            )
            fig.add_vline(
                x=GMF * i + speed_real * (18 - j),
                line_dash="dot",
                line_color="firebrick",
                line_width=1.5,
            )

    # GMF 2
    speed_real_2 = speed_real / first_gear_ratio_MCC5
    GMF2 = speed_real_2 * 29
    for i in range(1, 3):
        fig.add_vline(x=GMF2 * i, line_dash="dash", line_color="green")

        for j in range(1, 3):
            fig.add_vline(
                x=GMF2 * i - speed_real_2 * j,
                line_dash="dash",
                line_color="black",
                line_width=1,
                opacity=0.3,
            )
            fig.add_vline(
                x=GMF2 * i + speed_real_2 * j,
                line_dash="dash",
                line_color="black",
                line_width=1,
                opacity=0.3,
            )

    return fig


def order_track_signal_MCC5(
    sig, tacho, time, num_samples_per_revolution=samples_per_revolution_MCC5
):
    tacho_threshold = 3
    # num_samples_per_revolution = 360
    tacho_component_ratio = 29 / 95

    # Tachometer leading edge used as reference
    crossings = np.where(
        np.logical_and(tacho[:-1] <= tacho_threshold, tacho[1:] > tacho_threshold)
    )[0]

    # Angles at crossings
    crossing_angles = np.arange(len(crossings)) * 360
    # Angles at fault location
    crossing_angles_at_fault = crossing_angles / tacho_component_ratio

    # Only keep crossing angles at fault component within the range of original crossings
    crossing_angles_at_fault = crossing_angles_at_fault[
        crossing_angles_at_fault <= crossing_angles[-1]
    ]

    # Interpolate to find times at fault crossings
    t_crossings_at_fault = np.interp(
        crossing_angles_at_fault, crossing_angles, time[crossings]
    )

    # Essentially upsamples the time crossings to get the order tracked time points
    ot_interpolation_times = np.interp(
        np.arange((len(t_crossings_at_fault) - 1) * num_samples_per_revolution),
        np.arange(len(t_crossings_at_fault)) * num_samples_per_revolution,
        t_crossings_at_fault,
    )

    # Interpolate signal to new time points
    cs = CubicSpline(time, sig)
    ot_signal = cs(ot_interpolation_times)

    return ot_signal, ot_interpolation_times

In [ ]:
mes_H_MCC5 = read_MCC5_hive(
    MCC5_processed_data_dir / "MCC5_hive",
    fault="healthy",
    # severity="M",
    circulation="speed",
    section=0,
    speed_nominal=speed_nominal_MCC5,
    load_nominal=load_nominal_MCC5,
)  # [: sampling_freq_MCC5 * 10]

print(mes_H_MCC5.head(2))
print(mes_H_MCC5["speed"][0])
fig_M = plot_MCC5_FFT(mes_H_MCC5, sideband_block_size=8)
fig_M

In [ ]:
mes_L_MCC5 = read_MCC5_hive(
    MCC5_processed_data_dir / "MCC5_hive",
    # fault="crack",
    fault="missing_tooth",
    # fault="crack",
    # severity="L",
    circulation="speed",
    section=0,
    speed_nominal=speed_nominal_MCC5,
    load_nominal=load_nominal_MCC5,
)  # [: sampling_freq_MCC5 * 10]

print(mes_L_MCC5.head(2))
print(mes_L_MCC5["speed"][0])
fig_L = plot_MCC5_FFT(mes_L_MCC5, sideband_block_size=8)
fig_L

In [ ]:
mes_WEAR_MCC5 = read_MCC5_hive(
    MCC5_processed_data_dir / "MCC5_hive",
    fault="wear",
    severity="L",
    circulation="speed",
    section=0,
    speed_nominal=speed_nominal_MCC5,
    load_nominal=load_nominal_MCC5,
)  # [: sampling_freq_MCC5 * 10]

print(mes_WEAR_MCC5.head(2))
print(mes_WEAR_MCC5["speed"][0])
fig_WEAR = plot_MCC5_FFT(mes_WEAR_MCC5, sideband_block_size=8)
fig_WEAR

#### MCC5 OT


In [ ]:
# H
sig_H_MCC5 = mes_H_MCC5[sensor_MCC5].to_numpy()
tacho_H_MCC5 = mes_H_MCC5["tacho_in"].to_numpy()
time_H_MCC5 = np.arange(len(sig_H_MCC5)) / sampling_freq_MCC5

ot_H_MCC5, time_H_MCC5 = order_track_signal_MCC5(sig_H_MCC5, tacho_H_MCC5, time_H_MCC5)

# L
sig_L_MCC5 = mes_L_MCC5[sensor_MCC5].to_numpy()
tacho_L_MCC5 = mes_L_MCC5["tacho_in"].to_numpy()
time_L_MCC5 = np.arange(len(sig_L_MCC5)) / sampling_freq_MCC5

ot_L_MCC5, ot_time_L_MCC5 = order_track_signal_MCC5(
    sig_L_MCC5, tacho_L_MCC5, time_L_MCC5
)

# WEAR
sig_WEAR_MCC5 = mes_WEAR_MCC5[sensor_MCC5].to_numpy()
tacho_WEAR_MCC5 = mes_WEAR_MCC5["tacho_in"].to_numpy()
time_WEAR_MCC5 = np.arange(len(sig_WEAR_MCC5)) / sampling_freq_MCC5

ot_WEAR_MCC5, ot_time_WEAR_MCC5 = order_track_signal_MCC5(
    sig_WEAR_MCC5, tacho_WEAR_MCC5, time_WEAR_MCC5
)

go.Figure(
    px.line(x=time_L_MCC5, y=sig_L_MCC5).data
    + px
    .line(x=ot_time_L_MCC5, y=ot_L_MCC5)
    .update_traces(line_color="red", opacity=0.5)
    .data
)

In [ ]:
# Get crossings
crossings = np.where(np.logical_and(tacho_L_MCC5[:-1] <= 3, tacho_L_MCC5[1:] > 3))[0]

start_idx = crossings[0]
end_idx = crossings[-1]

crossings_angle = np.arange(len(crossings))  # * 360
input_shaft_position = np.interp(
    time_L_MCC5[start_idx:end_idx], time_L_MCC5[crossings], crossings_angle
)
shaft_position = input_shaft_position * (29 / 95)
shaft_position = shaft_position  # / 360

s_ot, new_sig = ordertrack_pyvib(
    time_L_MCC5[start_idx:end_idx],
    sig_L_MCC5[start_idx:end_idx],
    time_L_MCC5[start_idx:end_idx],
    shaft_position,
    1 / 360,
)

In [ ]:
# a = px.line(new_sig[s_ot < np.floor(s_ot[-1])])
a = px.line(new_sig)
a.add_scatter(y=ot_L_MCC5)

In [ ]:
s_fft = np.abs(np.fft.rfft(new_sig[s_ot < np.floor(s_ot[-1])], norm="forward"))
orig_fft = np.abs(np.fft.rfft(ot_L_MCC5, norm="forward"))

a = px.line(s_fft)
a.add_scatter(y=orig_fft)

In [ ]:
def ordertrack_pyvib2(t, y, tacho, ds, cubic=True):
    """
        Order track the vibration signal to match a desired position vector

        Parameters
        ----------
    t : float 1D array
        Time signal
    y : float 1D array
        Signal to order track
    ds : float
        Desired delta position
    cubic : boolean, optional
        Whether cubic interpolation is used

        Returns
        -------
    s_ot : float 1D array
        Shaft position of returned signal
    y_ot : float 1D array
        Order tracked signal
    """

    # Copy data
    t = np.array(t)
    y = np.array(y)
    t_s = t

    # NOTE: Start my changes
    crossings = np.where(np.logical_and(tacho[:-1] <= 3, tacho[1:] > 3))[0]

    start_idx = crossings[0]
    end_idx = crossings[-1]

    # Not actually angle but % of revolution
    crossings_angle = np.arange(len(crossings))
    input_shaft_position = np.interp(
        t[start_idx:end_idx], t[crossings], crossings_angle
    )
    s = input_shaft_position * (29 / 95)

    t = t[start_idx:end_idx]
    t_s = t_s[start_idx:end_idx]
    y = y[start_idx:end_idx]

    # NOTE: End my changes

    # Parameters
    dt = t[1] - t[0]

    # Get the overlapping parts between y and s
    temp = t[0]
    t = t - temp
    t_s = t_s - temp
    i1 = 0
    i2 = t.size
    if t_s[0] > 0.0:
        i1 = int(np.ceil(t_s[0] / dt))
    if t_s[-1] < t[-1]:
        i2 = int(np.floor(t_s[-1] / dt))
    t = t[i1:i2]
    y = y[i1:i2]

    # Interpolate the measured position to match vibration time
    f = interp1d(t_s, s)
    s = f(t)

    # Make desired position signal
    s_ot = np.arange(s[0], s[-1], ds)

    # print('s.size=%i, y.size=%i' % (s.size, y.size))

    # Make order tracked signal
    if cubic is True:
        f = InterpolatedUnivariateSpline(s, y, k=3)
    else:
        f = interp1d(s, y)
    y_ot = f(s_ot)

    s_ot = s_ot[s_ot < np.floor(s_ot[-1])]  # Only get full revolutions
    y_ot = y_ot[: s_ot.size]

    # Return data
    return s_ot, y_ot


s_ot, new_sig = ordertrack_pyvib2(
    time_L_MCC5,
    sig_L_MCC5,
    tacho_L_MCC5,
    1 / 360,
)

# a = px.line(new_sig[s_ot < np.floor(s_ot[-1])])
a = px.line(new_sig)
a.add_scatter(y=ot_L_MCC5)

In [ ]:
# Healthy
ot_fft_H_MCC5 = np.abs(
    np.fft.rfft(
        ot_H_MCC5[: 40 * samples_per_revolution_MCC5]
        .reshape(40, samples_per_revolution_MCC5)
        .mean(0),
        norm="forward",
    )
)
ot_fft_H_MCC5 = np.log(ot_fft_H_MCC5)
fig_ot_fft_H_MCC5 = px.line(y=ot_fft_H_MCC5)

# Large crack
ot_fft_L_MCC5 = np.abs(
    np.fft.rfft(
        ot_L_MCC5[: 40 * samples_per_revolution_MCC5]
        .reshape(40, samples_per_revolution_MCC5)
        .mean(0),
        norm="forward",
    )
)
ot_fft_L_MCC5 = np.log(ot_fft_L_MCC5)
fig_ot_fft_L_MCC5 = px.line(y=ot_fft_L_MCC5)

# Large wear
ot_fft_WEAR_MCC5 = np.abs(
    np.fft.rfft(
        ot_WEAR_MCC5[: 40 * samples_per_revolution_MCC5]
        .reshape(40, samples_per_revolution_MCC5)
        .mean(0),
        norm="forward",
    )
)
ot_fft_WEAR_MCC5 = np.log(ot_fft_WEAR_MCC5)
fig_ot_fft_WEAR_MCC5 = px.line(y=ot_fft_WEAR_MCC5)

# Combine
f_ot_fft_MCC5 = go.Figure(
    data=fig_ot_fft_H_MCC5.data
    + fig_ot_fft_L_MCC5.update_traces(line_color="red").data
    + fig_ot_fft_WEAR_MCC5.update_traces(line_color="green").data
)

sideband_block_size_lower = 5
sideband_block_size_upper = 8
for i in range(1, 8):
    f_ot_fft_MCC5.add_vline(x=i * num_teeth_MCC5, line_dash="dash")

    # for j in range(1, sideband_block_size_lower + 1):
    #     f_ot_fft_MCC5.add_vline(
    #         x=num_teeth_MCC5 * i - j,
    #         line_dash="dot",
    #         line_color="orange",
    #         line_width=1.5,
    #     )
    #     f_ot_fft_MCC5.add_vline(
    #         x=num_teeth_MCC5 * i + j,
    #         line_dash="dot",
    #         line_color="firebrick",
    #         line_width=1.5,
    #     )

    # for j in range(1, sideband_block_size_upper + 1):
    #     # Outer sidebands
    #     f_ot_fft_MCC5.add_vline(
    #         x=num_teeth_MCC5 * i - (18 - j),
    #         line_dash="dot",
    #         line_color="orange",
    #         line_width=1.5,
    #     )
    #     f_ot_fft_MCC5.add_vline(
    #         x=num_teeth_MCC5 * i + (18 - j),
    #         line_dash="dot",
    #         line_color="firebrick",
    #         line_width=1.5,
    # )

f_ot_fft_MCC5.show()

## AGFD

-   Journal: https://www.sciencedirect.com/science/article/pii/S2352340924011338
-   Mendeley Data: https://data.mendeley.com/datasets/fywnj597d8/2

**ERRORS**

-   `acc2` in installation 3 is pure noise, as the sensor is damaged. DO NOT USE!

**Notes:**

-   Sampling frequency 3012 Hz
-   `acc4` seems to work the best


In [ ]:
AGFD_processed_data_dir = Path.cwd().parent.parent.parent / "data" / "ROTOMATE" / "AGFD"

# # Read everything
# read_AGFD_hive(AGFD_processed_data_dir / "AGFD_hive").head(3)

# # Get one thing
# read_AGFD_hive(
#     AGFD_processed_data_dir / "AGFD_hive",
#     fault="crack",
#     severity="L",
#     installation=1,
#     speed_nominal=1000,
#     load_nominal=1,
# ).head(3)

# # Loop through all

# # Faults
# for fault in ["crack", "wear", "pitting"]:
#     for severity in ["S", "L"]:
#         for speed_nominal in [250, 500, 750, 1000, 1250, 1500]:
#             for load_nominal in [1, 6, 11]:
#                 for installation in [1, 2, 3]:
#                     measurement = read_AGFD_hive(
#                         AGFD_processed_data_dir / "AGFD_hive",
#                         fault=fault,
#                         severity=severity,
#                         installation=installation,
#                         speed_nominal=speed_nominal,
#                         load_nominal=load_nominal,
#                         drop_nominal=False,
#                     )
#                 # NOTE: Do something with `measurement` here

# # Healthy
# for GP in [1, 2, 3, 4, 5, 6, 7, 9]:
#     for speed_nominal in [250, 500, 750, 1000, 1250, 1500]:
#         for load_nominal in [1, 6, 11]:
#             measurement = read_AGFD_hive(
#                 AGFD_processed_data_dir / "AGFD_hive",
#                 fault="healthy",
#                 GP=GP,
#                 speed_nominal=speed_nominal,
#                 load_nominal=load_nominal,
#                 drop_nominal=False,
#             )
#         # NOTE: Do something with `measurement` here

In [ ]:
# Get one thing
mes_L_AGFD = read_AGFD_hive(
    AGFD_processed_data_dir / "AGFD_hive",
    fault="crack",
    severity="L",
    installation=1,
    speed_nominal=1000,
    load_nominal=6,
)
mes_L_AGFD.head(2)